In [1]:
import pandas as pd
df = pd.read_csv('ecommerce_sales_data.csv')
#df = df[df['variety'].notna()] # remove any NaN values as it blows up serialization
data = df.to_dict('records')
df

,Order Date,Product Name,Category,Region,Quantity,Sales,Profit
0,2024-12-31,Printer,Office,North,4,3640,348.93
1,2022-11-27,Mouse,Accessories,East,7,1197,106.53
2,2022-05-11,Tablet,Electronics,South,5,5865,502.73
3,2024-03-16,Mouse,Accessories,South,2,786,202.87
4,2022-09-10,Mouse,Accessories,West,1,509,103.28
...,...,...,...,...,...,...,...
3495,2023-02-15,Monitor,Accessories,North,4,4064,771.16
3496,2022-09-18,Monitor,Accessories,East,1,1117,119.89
3497,2022-04-12,Laptop,Electronics,South,4,260,66.02
3498,2022-01-18,Printer,Office,South,3,222,50.28


In [2]:
df.isnull().sum()

,0
Order Date,0
Product Name,0
Category,0
Region,0
Quantity,0
Sales,0
Profit,0


In [4]:
#!pip install qdrant-client
from qdrant_client import models, QdrantClient
from sentence_transformers import SentenceTransformer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 377.2/377.2 kB 7.8 MB/s eta 0:00:00


In [5]:
encoder = SentenceTransformer('all-MiniLM-L6-v2') # Model to create embeddings

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [20]:
# create the vector database client
qdrant = QdrantClient(":memory:") # Create in-memory Qdrant instance

In [21]:
# Create collection to store books
qdrant.recreate_collection(
    collection_name="sales",
    vectors_config=models.VectorParams(
        size=encoder.get_sentence_embedding_dimension(), # Vector size is defined by used model
        distance=models.Distance.COSINE
    )
)

/tmp/ipython-input-3522047366.py:2: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  qdrant.recreate_collection(


True

In [22]:
# vectorize!
qdrant.upload_points(
    collection_name="sales",
    points=[
        models.PointStruct(
            id=idx,
            vector=encoder.encode(doc["Product Name"]).tolist(),
            payload=doc
        ) for idx, doc in enumerate(data)
    ]
)

In [23]:
# Search time for sales data!

hits = qdrant.search(
    collection_name="sales",
    query_vector=encoder.encode("which is the popular product").tolist(),
    limit=3
)
for hit in hits:
  print(hit.payload, "score:", hit.score)

AttributeError: 'QdrantClient' object has no attribute 'search'

In [31]:
# Search time for sales data!

hits = qdrant.query_points(
    collection_name="sales",
    query=encoder.encode("best product").tolist(),
    limit=3
)
for hit in hits.points:
  print(hit.payload, "score:", hit.score)

{'Order Date': '2023-04-18', 'Product Name': 'Tablet', 'Category': 'Electronics', 'Region': 'South', 'Quantity': 7, 'Sales': 2709, 'Profit': 702.79} score: 0.368639351663681
{'Order Date': '2023-01-21', 'Product Name': 'Tablet', 'Category': 'Electronics', 'Region': 'North', 'Quantity': 1, 'Sales': 607, 'Profit': 33.77} score: 0.368639351663681
{'Order Date': '2022-06-02', 'Product Name': 'Tablet', 'Category': 'Electronics', 'Region': 'West', 'Quantity': 7, 'Sales': 847, 'Profit': 154.05} score: 0.368639351663681


In [33]:
search_results = [hit.payload for hit in hits.points]

In [34]:
# Now time to connect to the local large language model
from openai import OpenAI
client = OpenAI(
    base_url="http://127.0.0.1:8080/v1", # "http://<Your api-server IP>:port"
    api_key = "sk-no-key-required"
)
completion = client.chat.completions.create(
    model="LLaMA_CPP",
    messages=[
        {"role": "system", "content": "You are chatbot, a wine specialist. Your top priority is to help guide users into selecting amazing wine and guide them with their requests."},
        {"role": "user", "content": "Suggest me an amazing Malbec wine from Argentina"},
        {"role": "assistant", "content": str(search_results)}
    ]
)
print(completion.choices[0].message)

NotFoundError: Error code: 404